# Create CEOs Dataset

In [6]:
! pip install openpyxl pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 40.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Read original Excel dataset

In [ ]:
import pandas as pd

columns = ["symbol", "name", "CEO_name", "twitter_handle", "link", "sector", "year_extracted"]
df = pd.read_excel("data/S&P 500 CEO Data.xlsx", header=None)
df = df.iloc[1:]
df.columns = columns
df = df.reset_index(drop=True)
df.to_csv("data/ceos.csv", index=False, encoding="utf-8-sig")

print(df.head())

## CEOs list from Wikipedia

In [8]:
import pandas as pd

link = (
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies#S&P_500_component_stocks"
)

# Add headers to mimic a browser
df = pd.read_html(
    link,
    header=0,
    storage_options={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
)[0]

df.to_csv("data/constituents.csv", index=False)

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

def get_wikipedia_url(company_name):
    """Convert company name to Wikipedia URL format"""
    # Remove common suffixes and clean up
    cleaned = company_name.replace(" Inc.", "").replace(" Corporation", "").replace(" Corp.", "")
    cleaned = cleaned.replace(" Company", "").replace(" Co.", "").replace(",", "")
    # Replace spaces with underscores
    url_name = cleaned.strip().replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{url_name}"

def extract_ceo_from_infobox(soup):
    """Extract CEO name(s) from Wikipedia infobox"""
    ceos = []

    # Find the infobox
    infobox = soup.find('table', {'class': 'infobox'})
    if not infobox:
        return None

    # Look for rows with "Key people" or similar labels
    rows = infobox.find_all('tr')
    for row in rows:
        header = row.find('th')
        if header:
            header_text = header.get_text().strip().lower()
            if 'key people' in header_text or 'key person' in header_text:
                data = row.find('td')
                if data:
                    # Extract text and look for CEO mentions
                    text = data.get_text()
                    # Split by newlines to handle multiple people
                    lines = text.split('\n')
                    for line in lines:
                        # Look for CEO, Chief Executive Officer, President and CEO, etc.
                        if re.search(r'\b(CEO|Chief Executive Officer)\b', line, re.IGNORECASE):
                            # Extract the name (usually before the title in parentheses)
                            # Pattern: "Name (Title)" or "Name, Title"
                            name_match = re.match(r'^([^(\[]+)', line)
                            if name_match:
                                name = name_match.group(1).strip()
                                # Clean up common artifacts
                                name = re.sub(r'\[.*?\]', '', name)  # Remove citations
                                name = name.strip(' ,')
                                if name and len(name) > 2:  # Avoid empty or very short matches
                                    ceos.append(name)

    return ', '.join(ceos) if ceos else None

def scrape_ceo_data(input_csv, output_csv, delay=1):
    """
    Scrape CEO data for companies in the input CSV

    Args:
        input_csv: Path to input CSV file
        output_csv: Path to output CSV file
        delay: Delay between requests in seconds (be nice to Wikipedia!)
    """
    # Read existing data
    df = pd.read_csv(input_csv)

    # Check if we're resuming - look for existing output file
    try:
        existing_df = pd.read_csv(output_csv)
        print(f"Found existing progress file with {len(existing_df)} rows")
        # Add columns if they don't exist
        if 'wikipedia_url' not in existing_df.columns:
            existing_df['wikipedia_url'] = None
        if 'CEO' not in existing_df.columns:
            existing_df['CEO'] = None
        df = existing_df
        start_idx = len(existing_df)
    except FileNotFoundError:
        # Add new columns
        df['wikipedia_url'] = None
        df['CEO'] = None
        start_idx = 0
        print("Starting fresh scrape")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    total = len(df)

    for idx in range(start_idx, total):
        company_name = df.loc[idx, 'Security']
        symbol = df.loc[idx, 'Symbol']

        print(f"\n[{idx+1}/{total}] Processing: {company_name} ({symbol})")

        # Generate Wikipedia URL
        wiki_url = get_wikipedia_url(company_name)
        df.loc[idx, 'wikipedia_url'] = wiki_url

        try:
            # Fetch the page
            response = requests.get(wiki_url, headers=headers, timeout=10)

            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                ceo = extract_ceo_from_infobox(soup)

                if ceo:
                    df.loc[idx, 'CEO'] = ceo
                    print(f"  ✓ Found CEO: {ceo}")
                else:
                    print(f"  ✗ No CEO found in infobox")
            else:
                print(f"  ✗ HTTP {response.status_code} - Page not found or inaccessible")

        except requests.exceptions.RequestException as e:
            print(f"  ✗ Request failed: {str(e)[:50]}")

        except Exception as e:
            print(f"  ✗ Unexpected error: {str(e)[:50]}")

        # Save progress after each company
        df.to_csv(output_csv, index=False)

        # Be polite - wait between requests
        if idx < total - 1:  # Don't wait after the last one
            time.sleep(delay)

    print(f"\n✓ Scraping complete! Results saved to {output_csv}")
    print(f"  Found CEOs for {df['CEO'].notna().sum()} out of {total} companies")

    return df

# Run the scraper
if __name__ == "__main__":
    input_file = "data/constituents.csv"
    output_file = "data/constituents_with_ceos.csv"

    print("Starting S&P 500 CEO scraper...")
    print("This will take approximately 8-10 minutes for 500 companies")
    print("Progress is saved after each company, so you can stop and resume anytime\n")

    result_df = scrape_ceo_data(input_file, output_file, delay=1)

    # Show summary
    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total companies: {len(result_df)}")
    print(f"CEOs found: {result_df['CEO'].notna().sum()}")
    print(f"CEOs not found: {result_df['CEO'].isna().sum()}")

Starting S&P 500 CEO scraper...
This will take approximately 8-10 minutes for 500 companies
Progress is saved after each company, so you can stop and resume anytime

Starting fresh scrape

[1/503] Processing: 3M (MMM)
  ✓ Found CEO: William M. Brown

[2/503] Processing: A. O. Smith (AOS)
  ✓ Found CEO: Kevin J. Wheeler

[3/503] Processing: Abbott Laboratories (ABT)
  ✓ Found CEO: Robert B. Ford

[4/503] Processing: AbbVie (ABBV)
  ✓ Found CEO: Richard A. Gonzalez

[5/503] Processing: Accenture (ACN)
  ✓ Found CEO: Julie Sweet

[6/503] Processing: Adobe Inc. (ADBE)
  ✗ No CEO found in infobox

[7/503] Processing: Advanced Micro Devices (AMD)
  ✗ HTTP 403 - Page not found or inaccessible

[8/503] Processing: AES Corporation (AES)
  ✗ HTTP 403 - Page not found or inaccessible

[9/503] Processing: Aflac (AFL)
  ✗ HTTP 403 - Page not found or inaccessible

[10/503] Processing: Agilent Technologies (A)
  ✓ Found CEO: Padraig McDonnell

[11/503] Processing: Air Products (APD)
  ✗ HTTP 403 - P

# Fortune 500 CEOs

In [6]:
import pandas as pd

df = pd.read_csv("data/fortune_500_ceos.csv")

print(f"Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
print(f"Columns: {df.columns.tolist()}")
print(df.iloc[309])

Number of rows: 500
Number of columns: 10
Columns: ['rank', 'company', 'industry', 'city', 'state', 'zip', 'website', 'employees', 'revenue', 'CEO']

First 3 rows:
rank                                      310
company          Western & Southern Financial
industry     Insurance: Life, Health (Mutual)
city                               Cincinnati
state                                    Ohio
zip                                     45202
website                   westernsouthern.com
employees                               2,744
revenue                      $ 13,769,000,000
CEO                              John Barrett
Name: 309, dtype: object
